# Обучение классификатора обращений абитуриентов

В этом ноутбуке мы дообучаем русскоязычную BERT-модель (`DeepPavlov/rubert-base-cased`) для классификации входящих обращений по 12 категориям приёмной комиссии.

## Категории

| Slug | Описание |
|---|---|
| `tech_issue` | Технические проблемы с личным кабинетом |
| `deadlines` | Сроки и этапы приёмной кампании |
| `docs_submission` | Подача документов и исправление ошибок |
| `status_check` | Отслеживание статуса заявления |
| `admission_scores` | Проходные баллы и конкурс |
| `finance_contracts` | Оплата, договоры, стоимость обучения |
| `enrollment` | Зачисление, согласие, приказы |
| `dormitory` | Общежитие и проживание |
| `academic_info` | Расписание, студенческий билет, начало учёбы |
| `events` | Дни открытых дверей и мероприятия |
| `general_info` | Общие контакты и адрес университета |
| `program_consult` | Выбор направления и программы |

## Предварительные требования

Перед запуском ноутбука убедитесь, что:
1. Данные сгенерированы: `python data/scripts/generate_deepseek.py`
2. Данные предобработаны: `python data/scripts/preprocess.py`
3. Установлены зависимости: `pip install -r requirements.txt`

## 0. Импорты и конфигурация

In [2]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.dpi'] = 120

# Корень проекта
ROOT = Path("..")
PROCESSED_DIR = ROOT / "data" / "processed"
MODEL_SAVE_PATH = ROOT / "models" / "rubert-ticket-classifier"
BASE_MODEL = "DeepPavlov/rubert-base-cased"

print('STARTING')

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Устройство: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

STARTING
Устройство: cuda
GPU: NVIDIA GeForce RTX 5060 Ti


## 1. Загрузка данных

Загружаем предобработанные выборки из `data/processed/`. Каждая запись содержит:
- `text` — текст обращения абитуриента
- `label` — числовой индекс категории
- `category` — slug категории (для отчётов)

In [3]:
def load_split(name: str) -> list:
    path = PROCESSED_DIR / f"{name}.json"
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train_data = load_split("train")
val_data   = load_split("val")
test_data  = load_split("test")

with open(PROCESSED_DIR / "label_map.json", "r", encoding="utf-8") as f:
    label_map: dict = json.load(f)  # slug -> index

idx_to_label = {v: k for k, v in label_map.items()}
num_labels = len(label_map)

print(f"Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}")
print(f"Количество классов: {num_labels}")
print(f"\nПример записи:")
print(json.dumps(train_data[0], ensure_ascii=False, indent=2))

FileNotFoundError: [Errno 2] No such file or directory: '..\\data\\processed\\train.json'

### Распределение классов в обучающей выборке

In [ ]:
from collections import Counter

counts = Counter(s["category"] for s in train_data)
labels_sorted = sorted(counts.keys(), key=lambda x: -counts[x])
values = [counts[l] for l in labels_sorted]

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(labels_sorted, values, color="steelblue")
ax.set_xlabel("Количество примеров")
ax.set_title("Распределение категорий в обучающей выборке")
plt.tight_layout()
plt.show()

## 2. Предобработка — токенизация

Загружаем токенизатор `DeepPavlov/rubert-base-cased` и преобразуем тексты в тензоры.
Максимальная длина последовательности — 256 токенов (достаточно для коротких обращений).

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

MAX_LEN = 256

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
    )

def make_dataset(data: list) -> Dataset:
    ds = Dataset.from_list([{"text": s["text"], "labels": s["label"]} for s in data])
    return ds.map(tokenize, batched=True, remove_columns=["text"])

train_ds = make_dataset(train_data)
val_ds   = make_dataset(val_data)
test_ds  = make_dataset(test_data)

train_ds.set_format("torch")
val_ds.set_format("torch")
test_ds.set_format("torch")

print(f"Токенизация завершена. Пример признаков: {list(train_ds.features.keys())}")

## 3. Обучение модели

Инициализируем `rubert-base-cased` с классификационной головой на 12 классов.

### Параметры обучения

| Параметр | Значение |
|---|---|
| Эпохи | 5 |
| Batch size (train) | 16 |
| Batch size (eval) | 32 |
| Learning rate | 2e-5 |
| Weight decay | 0.01 |
| Метрика выбора | accuracy |

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL,
    num_labels=num_labels,
    id2label=idx_to_label,
    label2id=label_map,
)

print(f"Параметров модели: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    accuracy = (preds == labels).mean()
    return {"accuracy": float(accuracy)}


training_args = TrainingArguments(
    output_dir=str(MODEL_SAVE_PATH / "checkpoints"),
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
    fp16=(device == "cuda"),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("Начало обучения...")
trainer.train()

## 4. Оценка качества на тестовой выборке

После обучения оцениваем модель на отложенной тестовой выборке.
Выводим точность по каждой категории, macro F1 и weighted F1.

In [ ]:
test_preds = trainer.predict(test_ds)
pred_labels = np.argmax(test_preds.predictions, axis=-1)
true_labels = test_preds.label_ids

target_names = [idx_to_label[i] for i in range(num_labels)]

print("=== Отчёт по классификации (тестовая выборка) ===")
print(classification_report(
    true_labels, pred_labels,
    target_names=target_names,
    zero_division=0,
))

## 5. Анализ ошибок — матрица ошибок

Матрица ошибок показывает, какие категории модель путает между собой.
Идеальная модель — диагональная матрица.

In [ ]:
cm = confusion_matrix(true_labels, pred_labels)

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=target_names,
    yticklabels=target_names,
    ax=ax,
)
ax.set_xlabel("Предсказано")
ax.set_ylabel("Истинная метка")
ax.set_title("Матрица ошибок на тестовой выборке")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Самые проблемные категории (низкий recall)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precision, recall, f1, support = precision_recall_fscore_support(
    true_labels, pred_labels, zero_division=0
)

stats = sorted(
    zip(target_names, precision, recall, f1, support),
    key=lambda x: x[2],  # сортировка по recall
)

print(f"{'Категория':<22} {'Precision':>9} {'Recall':>8} {'F1':>6} {'Support':>8}")
print("-" * 58)
for name, p, r, f, s in stats:
    flag = " ⚠" if r < 0.6 else ""
    print(f"{name:<22} {p:>9.3f} {r:>8.3f} {f:>6.3f} {s:>8}{flag}")

## 6. Сохранение модели

Сохраняем модель и токенизатор в `models/rubert-ticket-classifier/`.
Также сохраняем `label_map.json` рядом с моделью, чтобы `classifier.py` мог
восстановить соответствие индекс → slug без дополнительной конфигурации.

После сохранения добавьте в `.env`:
```
USE_FINE_TUNED_MODEL=True
FINE_TUNED_MODEL_PATH=models/rubert-ticket-classifier
```

In [ ]:
MODEL_SAVE_PATH.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(MODEL_SAVE_PATH))
tokenizer.save_pretrained(str(MODEL_SAVE_PATH))

# Сохраняем карту меток рядом с моделью
with open(MODEL_SAVE_PATH / "label_map.json", "w", encoding="utf-8") as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)

print(f"Модель сохранена в: {MODEL_SAVE_PATH.resolve()}")
print("\nСодержимое директории:")
for p in sorted(MODEL_SAVE_PATH.iterdir()):
    if p.name != "checkpoints":
        print(f"  {p.name}  ({p.stat().st_size // 1024:,} KB)")

## 7. Быстрая проверка на примерах

Прогоняем несколько ручных примеров через сохранённую модель,
чтобы убедиться в корректности работы до деплоя.

In [ ]:
from transformers import pipeline as hf_pipeline

clf = hf_pipeline(
    "text-classification",
    model=str(MODEL_SAVE_PATH),
    tokenizer=str(MODEL_SAVE_PATH),
    device=0 if device == "cuda" else -1,
)

test_examples = [
    ("Не могу войти в личный кабинет, выдаёт ошибку 500", "tech_issue"),
    ("Когда последний день подачи оригиналов документов?", "deadlines"),
    ("Как оплатить первый семестр и получить договор?", "finance_contracts"),
    ("Мне пришло приглашение на зачисление, что делать дальше?", "enrollment"),
    ("Есть ли места в общежитии для иногородних?", "dormitory"),
]

print(f"{'Текст':<55} {'Ожидалось':<22} {'Предсказано':<22} {'Conf':>6}")
print("-" * 110)
for text, expected in test_examples:
    result = clf(text, truncation=True, max_length=256)[0]
    predicted = result["label"]
    conf = result["score"]
    mark = "✓" if predicted == expected else "✗"
    short_text = text[:52] + "..." if len(text) > 52 else text
    print(f"{short_text:<55} {expected:<22} {predicted:<22} {conf:>6.3f} {mark}")